In [ ]:
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager


def iniciar_navegador():
    options = Options()
    options.binary_location = "/usr/bin/google-chrome"
    options.add_argument("--headless")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--disable-gpu")
    options.add_argument("--disable-blink-features=AutomationControlled")

    return webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=options
    )

URL_OBJETIVO = "https://seguimientoejecucionobras.coordinador.cl/"

driver = iniciar_navegador()
dataset_final = []

try:
    print("Conectando al Coordinador Eléctrico...")
    driver.get(URL_OBJETIVO)

    time.sleep(10)  # importante para carga dinámica

    # Buscar filas de tablas
    filas = driver.find_elements(By.TAG_NAME, "tr")
    print(f"Filas encontradas: {len(filas)}")

    for fila in filas:
        columnas = fila.find_elements(By.TAG_NAME, "td")

        if len(columnas) >= 2:
            dato1 = columnas[0].text.strip()
            dato2 = columnas[1].text.strip()

            if dato1 and dato2:
                dataset_final.append({
                    "proyecto": dato1,
                    "detalle": dato2
                })

    if dataset_final:
        df = pd.DataFrame(dataset_final)
        archivo = "datos_trazabilidad_energia.csv"
        df.to_csv(archivo, index=False)

        print(f"Datos guardados en: {archivo}")
        print(df.head())

    else:
        print("No se capturaron datos. Puede ser contenido dinámico.")

except Exception as e:
    print(f"Error: {e}")

finally:
    driver.quit()
    print("Navegador cerrado")

Conectando al Coordinador Eléctrico...
